# Notebook 2 of 3: Downstream analysis for Stereo-seq

**What this notebook does.** It starts from the raw `.h5ad` that Notebook 1 produced (cells with gene counts and
positions) and runs the downstream analysis: quality filtering, clustering, cell type annotation, and tissue
domains with BANKSY.

**This is a general pipeline, not a keloid one.** The core steps and BANKSY work for any Stereo-seq sample. You
get cell types in one of two ways:
- **Name the clusters from markers** (Step 9, the generic annotation cell). You look at the marker genes and label
  each cluster. Works for any tissue once you swap in your markers.
- **Or map them automatically with cell2location** (Notebook 3, on a GPU). You give it an annotated reference for
  your tissue and it labels the cells for you.

**About Part B.** Part B (Step 10 onward) is an **optional worked example on keloid skin**. It shows a full real
analysis: fibroblast states, a collagen control, plasma cell foci, and a reference check. The method is reusable,
but the gene lists and labels are keloid specific. For your own tissue, either rewrite those pieces for your
biology, or skip Part B and go from the generic annotation (Step 9) straight to BANKSY (Step 16).

**How to run.** Click a cell, press **Shift+Enter**. Go top to bottom, in order. Edit only the lines marked
`# EDIT`. Several steps ask you to look at a plot, set a number in an earlier cell, and rerun. That loop is normal
and is called out where it happens.

**Set up once (in a terminal):**

Check which env has scanpy + BANKSY, paste this loop:
```bash

for e in $(conda env list | awk 'NF && $1!="#" {print $1}'); do
  printf '%-18s ' "$e"
  conda run -n "$e" python -c "import scanpy; from banksy.initialize_banksy import initialize_banksy; print('scanpy '+scanpy.__version__+' + BANKSY  <== use this env')" 2>/dev/null \
    || echo "no (missing scanpy and/or BANKSY)"
done

```

Note: the env that prints `<== use this env` is the one to launch Jupyter from and to pick as the kernel.

Then:

```bash
conda activate banksy         # EDIT the env name that has scanpy + BANKSY
jupyter lab                   # then pick the 'banksy' kernel, top right
```

# Part A.

## Step 1. Settings and load

Set the sample name and the three paths, then run. This loads the raw `.h5ad`, flags mitochondrial and ribosomal
genes, and computes the per-cell quality numbers used from here on. `counts raw? max =` should be a whole number
(the counts are integers at this point, not yet normalised).

You set the QC cutoffs at Step 2, right where the histograms are, so there is nothing to set here for them.

In [ ]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import scipy.sparse as sp
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
sc.settings.verbosity = 1


SAMPLE = "Y40172EA"                                                    # EDIT: sample name
IN  = Path("/Users/chobani/Downloads/Y40172EA_cpsam_proseg_raw.h5ad")  # EDIT: raw h5ad from Notebook 1
OUT = Path("/Users/chobani/Downloads/Y40172EA_out2"); OUT.mkdir(exist_ok=True)   # EDIT: output folder

sc.settings.figdir = OUT       # scanpy's save= figures land in the output folder

adata = sc.read_h5ad(IN); adata.var_names_make_unique()
adata.layers["counts"] = adata.X.copy()
adata.var["mt"]   = adata.var_names.str.upper().str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.upper().str.match(r"^RP[SL]")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt","ribo"], percent_top=None, inplace=True)
adata.obs["counts_per_volume"] = adata.obs["total_counts"] / adata.obs["volume"]
print(adata)
print("counts raw? max =", adata.X.max())

## Step 2. Quality cutoffs: look, then set, then check

Work through the three cells below in order.

1. **Look.** The first cell draws four histograms: counts per cell, genes per cell, percent mitochondrial, and
   cell volume. Counts and genes are on a log10 axis, so real cells form the tall bell and low quality cells sit
   as a small bump to its left.
2. **Set.** The second cell is where you type your three cutoffs, from what you just saw. Put `MIN_COUNTS` and
   `MIN_GENES` in the dip between the small left bump and the main bell. Put `MAX_PCT_MT` to the right of the main
   mitochondrial bump. Run it to save the numbers.
3. **Check.** The third cell redraws the histograms with your cutoff lines on them and prints how many cells each
   cutoff would remove. If the lines or the counts are not what you want, change the numbers in the set cell and
   run the check again.

Nothing is removed here. The filtering happens in Step 4, using the numbers you set.

In [ ]:
# 1. LOOK: the four quality distributions (no cutoffs yet, just the shapes)
import numpy as np
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].hist(np.log10(adata.obs["total_counts"] + 1), bins=100, color="0.4"); ax[0].set_title("counts / cell (log10)")
ax[1].hist(np.log10(adata.obs["n_genes_by_counts"] + 1), bins=100, color="0.4"); ax[1].set_title("genes / cell (log10)")
ax[2].hist(adata.obs["pct_counts_mt"], bins=100, color="0.4"); ax[2].set_title("% mitochondrial")
ax[3].hist(np.log10(adata.obs["volume"] + 1), bins=100, color="0.4"); ax[3].set_title("cell volume (log10)")
plt.tight_layout(); plt.show()

In [ ]:
# 2. SET: type your three cutoffs from the histograms above, then run this cell to save them.
MIN_COUNTS = 150   # EDIT: drop cells with fewer total counts than this
MIN_GENES  = 100   # EDIT: drop cells with fewer distinct genes than this
MAX_PCT_MT = 5     # EDIT: drop cells above this percent mitochondrial
print(f"cutoffs saved: MIN_COUNTS={MIN_COUNTS}, MIN_GENES={MIN_GENES}, MAX_PCT_MT={MAX_PCT_MT}")

In [ ]:
# 3. CHECK: the same histograms with your cutoff lines, and how many cells each would remove.
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(np.log10(adata.obs["total_counts"] + 1), bins=100, color="0.4")
ax[0].axvline(np.log10(MIN_COUNTS), color="r"); ax[0].set_title(f"counts (log10)  line = {MIN_COUNTS}")
ax[1].hist(np.log10(adata.obs["n_genes_by_counts"] + 1), bins=100, color="0.4")
ax[1].axvline(np.log10(MIN_GENES), color="r"); ax[1].set_title(f"genes (log10)  line = {MIN_GENES}")
ax[2].hist(adata.obs["pct_counts_mt"], bins=100, color="0.4")
ax[2].axvline(MAX_PCT_MT, color="r"); ax[2].set_title(f"% mito  line = {MAX_PCT_MT}")
plt.tight_layout(); plt.show()

n = adata.n_obs
lo_c = int((adata.obs["total_counts"] < MIN_COUNTS).sum())
lo_g = int((adata.obs["n_genes_by_counts"] < MIN_GENES).sum())
hi_m = int((adata.obs["pct_counts_mt"] >= MAX_PCT_MT).sum())
print(f"MIN_COUNTS = {MIN_COUNTS}: would remove {lo_c:,} cells ({100*lo_c/n:.1f}%)")
print(f"MIN_GENES  = {MIN_GENES}: would remove {lo_g:,} cells ({100*lo_g/n:.1f}%)")
print(f"MAX_PCT_MT = {MAX_PCT_MT}: would remove {hi_m:,} cells ({100*hi_m/n:.1f}%)")
print("nothing removed yet, that happens in Step 4")

## Step 3. Find segmentation mergers (two cells fused into one)

A real cell runs one lineage programme at a time. When segmentation accidentally fuses two neighbouring cells of
different lineages (say an epithelial cell and a fibroblast), the fused object shows both programmes at once. This
step measures, for each cell, the fraction of its counts coming from each lineage's markers, flags cells positive
for two or more lineages, and maps where they are. Those flagged cells are removed in Step 4.

**Make it yours: how to edit `progs`.** `progs` is a dictionary, one entry per major cell lineage in your tissue,
each with a few strong, specific marker genes:

```python
progs = {
    "lineage name": ["MARKER1", "MARKER2", "MARKER3"],
    ...
}
```

- List the **major lineages** you expect, three to seven is plenty. More lineages means more fusion types get caught.
- Give each **three to five canonical, specific** markers. You do not need many, they are summed as a fraction.
- **Case matters.** Human genes are uppercase (`COL1A1`), mouse are title case (`Col1a1`). The cell prints how many
  of your markers it found, so a `0/...` means the names did not match your data.

A mouse lung example:

```python
progs = {
    "epithelial":  ["Epcam", "Sftpc", "Scgb1a1"],
    "endothelial": ["Pecam1", "Cldn5", "Cdh5"],
    "fibroblast":  ["Col1a1", "Col3a1", "Pdgfra"],
    "immune":      ["Ptprc", "Lyz2", "Cd68"],
    "muscle":      ["Acta2", "Myh11", "Tagln"],
}
```

`THR = 0.02` means a lineage counts as present when its markers are at least 2 percent of the cell's counts. If
nothing gets flagged, either your sample is clean or your markers did not match, so check the counts it prints.

In [ ]:
# Flag likely segmentation mergers: cells showing two different lineage programmes at once.
# EDIT progs for your tissue. Defaults are broad HUMAN markers.
# For mouse use the mouse spelling, e.g. Col1a1, Pecam1, Ptprc, Epcam.
progs = {
    "epithelial":  ["EPCAM","KRT8","KRT18","KRT5","KRT14"],
    "fibroblast":  ["COL1A1","COL1A2","COL3A1","DCN","PDGFRA"],
    "endothelial": ["PECAM1","VWF","CLDN5","KDR"],
    "immune":      ["PTPRC","CD3E","LYZ","CD68","MS4A1"],
    "muscle":      ["ACTA2","MYH11","TAGLN","DES"],
}
THR = 0.02      # EDIT: a lineage counts as present when its markers are >= this fraction of the cell's counts

# fraction of a cell's counts that come from a gene set
def prog_frac(ad, genes):
    genes = [g for g in genes if g in ad.var_names]
    if not genes:
        return np.zeros(ad.n_obs)
    X = ad[:, genes].layers["counts"]
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    return X.sum(1) / np.maximum(ad.obs["total_counts"].values, 1)

# how many of your markers were actually found (this catches a species or spelling mismatch)
print("markers found per lineage:")
for nm, gs in progs.items():
    found = [g for g in gs if g in adata.var_names]
    print(f"  {nm:12s}: {len(found)}/{len(gs)}  {found}")
    if not found:
        print(f"      WARNING: none found for '{nm}', check the gene names for your species")

# lineage fraction per cell, then flag cells positive for two or more lineages
flags = pd.DataFrame(index=adata.obs_names)
for nm, gs in progs.items():
    adata.obs[f"f_{nm}"] = prog_frac(adata, gs)
    flags[nm] = adata.obs[f"f_{nm}"] > THR
adata.obs["n_programmes"] = flags.sum(1).values
adata.obs["merger_flag"]  = adata.obs["n_programmes"] >= 2
print(f"\nflagged as likely mergers: {int(adata.obs['merger_flag'].sum()):,} "
      f"({100*adata.obs['merger_flag'].mean():.1f}%)")

# where the flagged cells sit on the tissue
xy = adata.obsm["spatial"]; m = adata.obs["merger_flag"].values
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(xy[~m, 0], xy[~m, 1], s=1, c="0.85", linewidths=0)
ax.scatter(xy[m, 0],  xy[m, 1],  s=3, c="crimson", linewidths=0)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"likely segmentation mergers ({int(m.sum()):,})")
plt.show()

Optional check: how many cells clear no lineage at all, and how sensitive that is to the threshold. This helps
you judge whether `THR` is reasonable. Cells clearing no programme are usually low content or on the tissue edge,
not a distinct cell type. Safe to read and move on.

In [ ]:
pf = adata.obs[[f"f_{n}" for n in progs]]
adata.obs["max_prog"]  = pf.max(1).values
adata.obs["best_prog"] = pf.idxmax(1).str.replace("f_", "", regex=False).values
none = (adata.obs["n_programmes"] == 0).values

print("threshold sensitivity (cells clearing NO programme):")
for t in [0.02, 0.01, 0.005, 0.002]:
    print(f"  cut {t:<6}: {int((adata.obs['max_prog'] < t).sum()):>6,} ({100*(adata.obs['max_prog'] < t).mean():4.1f}%)")

print(f"\nmax programme fraction among the no-programme cells:")
print(adata.obs.loc[none, "max_prog"].describe().round(4).to_string())
print(f"\nmedian counts: none {adata.obs.loc[none,'total_counts'].median():.0f}  vs rest {adata.obs.loc[~none,'total_counts'].median():.0f}")
print(f"median volume: none {adata.obs.loc[none,'volume'].median():.0f}  vs rest {adata.obs.loc[~none,'volume'].median():.0f}")
print("\nclosest programme anyway:")
print(adata.obs.loc[none, "best_prog"].value_counts().to_string())

xy = adata.obsm["spatial"]
fig, ax = plt.subplots(figsize=(8,8))
ax.scatter(xy[~none,0], xy[~none,1], s=1, c="0.85", linewidths=0)
ax.scatter(xy[none,0],  xy[none,1],  s=1, c="darkorange", linewidths=0)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"no lineage programme above {int(THR*100)}% ({int(none.sum()):,}, {100*none.mean():.0f}%)")
plt.show()

## Step 4. Apply the filters

This drops the flagged mergers, then removes low count, low gene, and high mitochondrial cells using the cutoffs
you set in Step 2, and drops genes seen in fewer than 3 cells. It also records how many neighbours each cell has
within 50 microns, kept as a covariate so a sparse edge region is not later mistaken for a biological cluster. If
it removes too much or too little, change the cutoffs in Step 2 and rerun from there.

In [ ]:
from scipy.spatial import cKDTree

# ----------------------------------------------------------------------------------------------
# WHERE TO EDIT THE CUTOFFS: they are set in the SET cell of Step 2 (MIN_COUNTS, MIN_GENES, MAX_PCT_MT).
# To change what gets filtered, edit those three numbers in the Step 2 SET cell and
# rerun the Step 2 cells, then run this cell. The defaults are only a starting point, set them for your sample.
# ----------------------------------------------------------------------------------------------
print(f"filtering with MIN_COUNTS={MIN_COUNTS}, MIN_GENES={MIN_GENES}, MAX_PCT_MT={MAX_PCT_MT}\n")

# 1. drop the cells flagged as mergers in Step 3
before = adata.n_obs
adata = adata[~adata.obs["merger_flag"].values].copy()
print(f"dropped {before - adata.n_obs:,} flagged mergers")

# 2. drop low quality cells and rare genes, using the cutoffs from Step 2
b2 = adata.n_obs
sc.pp.filter_cells(adata, min_counts=MIN_COUNTS)              # too few total counts
sc.pp.filter_cells(adata, min_genes=MIN_GENES)                # too few distinct genes
adata = adata[adata.obs["pct_counts_mt"] < MAX_PCT_MT].copy() # too much mitochondrial
sc.pp.filter_genes(adata, min_cells=3)                        # genes seen in fewer than 3 cells
print(f"kept {adata.n_obs:,} / {b2:,} cells ({100*adata.n_obs/b2:.1f}%)")

# 3. record each cell's number of neighbours within 50 microns, kept as a covariate so a sparse
#    tissue edge is not later mistaken for a real cluster
tree = cKDTree(adata.obsm["spatial"])
adata.obs["n_within_50um"] = tree.query_ball_point(adata.obsm["spatial"], r=50, return_length=True)
print("neighbours within 50um: median", int(adata.obs["n_within_50um"].median()),
      "| 5th percentile", int(adata.obs["n_within_50um"].quantile(0.05)))

## Step 5. Normalise and pick the genes for clustering

This normalises the counts, keeps a log-normalised copy, and chooses the most informative genes with the
Pearson-residual method, which handles the low counts typical of Stereo-seq well. It then removes gene families
that would make cells cluster by a technical artefact rather than by biology: mitochondrial, ribosomal,
immunoglobulin, haemoglobin, long non-coding, histones, and unnamed clone or predicted genes. The name patterns
are matched on uppercased names, so they work for human and for mouse.

`N_HVG` sets how many candidate genes to consider. `AMBIENT` is an optional list for genes that are background
in your particular data (very long or sticky transcripts that leak everywhere); leave it empty unless you have
identified them. `kept for clustering` prints how many genes remain.

In [ ]:
# normalise, keep a log-normalised copy, then choose the genes used for clustering
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()
adata.raw = adata

N_HVG = 3000     # EDIT: how many candidate variable genes to consider
sc.experimental.pp.highly_variable_genes(adata, flavor="pearson_residuals", n_top_genes=N_HVG, layer="counts")

# gene families to drop from clustering. patterns are matched on UPPERCASED names, so they work
# for both human and mouse gene symbols.
vn = adata.var_names.str.upper()
technical = (
      vn.str.startswith("MT-")             # mitochondrial
    | vn.str.match(r"^RP[SL]")             # ribosomal
    | vn.str.match(r"^IG[HKL]")            # immunoglobulin
    | vn.str.startswith("HB")              # haemoglobin
    | vn.isin(["MALAT1", "NEAT1", "XIST"]) # very high, non-informative
    | vn.str.startswith("LINC")            # long non-coding (human)
    | vn.str.startswith("HIST")            # histones
    | vn.str.match(r"^(AC|AL|AD|FP|AP)\d") # unnamed clone accessions (human)
    | vn.str.match(r"^GM\d")               # predicted genes (mouse)
    | vn.str.contains(r"RIK$")             # Riken cDNAs (mouse)
)

# EDIT (optional): background/ambient genes specific to YOUR data. Leave empty unless you have found them.
# Example from one skin dataset: ["KIRREL3","IL1RAPL2","KCNMA1","ROBO2","DLC1","NCKAP5"]
AMBIENT = []
ambient = vn.isin([g.upper() for g in AMBIENT])

block = np.asarray(technical | ambient)
n_before = int(adata.var["highly_variable"].sum())
removed  = int((adata.var["highly_variable"].values & block).sum())
adata.var["highly_variable"] &= ~block
print(f"candidate variable genes: {n_before}  |  removed as technical/ambient: {removed}")
print("kept for clustering:", int(adata.var["highly_variable"].sum()), "genes")

## Step 6. PCA, and how many components to keep

This computes principal components on the Pearson-residual values of the chosen genes and draws the scree plot
(variance per component) and the cumulative curve. **Look at the scree plot:** find where it flattens into a
near-horizontal tail. Keep components up to about that point. The printed cumulative variance at 20, 30, 40
components helps. Set `N_PCS` in the next step accordingly.

In [ ]:
adp = adata[:, adata.var["highly_variable"]].copy()
adp.X = adp.layers["counts"].copy()
sc.experimental.pp.normalize_pearson_residuals(adp)
sc.pp.pca(adp, n_comps=50, random_state=0)
adata.obsm["X_pca"] = adp.obsm["X_pca"]; adata.uns["pca"] = adp.uns["pca"]; del adp

vr = adata.uns["pca"]["variance_ratio"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(1, len(vr)+1), vr, "-o", ms=3); ax[0].set_title("scree"); ax[0].set_xlabel("PC")
ax[1].plot(range(1, len(vr)+1), np.cumsum(vr), "-o", ms=3); ax[1].set_title("cumulative"); ax[1].set_xlabel("PC")
plt.tight_layout(); plt.show()
print("cumulative variance at 20/30/40:", [f"{np.cumsum(vr)[k-1]:.3f}" for k in (20,30,40)])

## Step 7. Cluster the cells and make the UMAP

This builds the neighbour graph, runs Leiden clustering, and computes the UMAP for viewing. **Two knobs:**
`N_PCS` (from Step 6) and `RES`, the resolution. Higher `RES` gives more, smaller clusters. Start at 1.0 and
adjust if clusters are too coarse or too fragmented. The UMAP is only a viewer; the clusters come from the graph,
not from the UMAP shape.

In [ ]:
N_PCS, RES = 20, 1.0        # EDIT: N_PCS from the scree plot in Step 6; RES higher = more clusters
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=N_PCS)
sc.tl.leiden(adata, resolution=RES, key_added="leiden", flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(adata)
print(adata.obs["leiden"].value_counts().to_string())
sc.pl.umap(adata, color="leiden", legend_loc="on data", frameon=False)

## Step 8. Marker genes per cluster

This ranks the genes that define each cluster and prints the top eight, alongside each cluster's median counts,
genes, volume, and local density. Nothing to edit here, it works for any sample. You use this table in Step 9 to
decide what each cluster is. A cluster with very low counts and volume across the board is usually low quality
rather than a real cell type.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
names = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(8)
for c in names.columns:
    print(f"cluster {int(c):>2}: {', '.join(names[c])}")
q = adata.obs.groupby("leiden")[["total_counts","n_genes_by_counts","volume","n_within_50um"]].median()
q["n"] = adata.obs["leiden"].value_counts()
print("\n", q.sort_values("total_counts").to_string())

## Step 9. Name the clusters (generic, do this for your tissue)

This is where the numbered Leiden clusters become named cell types. It works for any sample. The only two things
you change are your marker list and your cluster to type map.

**What to change for your sample:**
1. `broad_markers`: the major lineages you expect, each with a few specific markers. The defaults are common
   human genes. For mouse use title case (`Col1a1`, not `COL1A1`). Only the marker lists change, not the method.
2. `cluster_to_type`: the map from each cluster number to a cell type name. Leave it empty to accept the
   automatic best guess, or fill it in to correct the guess.

**What this cell does, in order:**
- prints the average expression of each lineage's markers per cluster (a table to guide you),
- makes an automatic best guess of the type for every cluster,
- applies your `cluster_to_type` (or the best guess if you left it empty) and draws the named UMAP.

Read the guide table together with the Step 8 marker lists, then fill `cluster_to_type`. This sets
`adata.obs["cell_type"]`, which BANKSY (Step 16) and the final save both use. If you would rather label
automatically from a reference instead, use Notebook 3 (cell2location), then continue from Step 16.

In [ ]:
# EDIT: broad lineage markers for YOUR tissue. Defaults are common human genes; for mouse use title case (Col1a1)
broad_markers = {
    "Epithelial":  ["EPCAM","KRT8","KRT18","KRT5","KRT14"],
    "Fibroblast":  ["COL1A1","COL1A2","DCN","LUM","PDGFRA"],
    "Endothelial": ["PECAM1","VWF","CLDN5","KDR"],
    "Mural":       ["RGS5","ACTA2","NOTCH3","PDGFRB"],
    "Myeloid":     ["LYZ","CD68","C1QA","AIF1"],
    "Lymphocyte":  ["CD3D","CD3E","CD8A","MS4A1","CD79A"],
    "Plasma":      ["JCHAIN","MZB1","IGKC"],
}

# average expression of each lineage's markers per cluster, to guide you
present = {k: [g for g in v if g in adata.raw.var_names] for k, v in broad_markers.items()}
cols = []
for lin, gs in present.items():
    if gs:
        s = sc.get.obs_df(adata, keys=gs + ["leiden"], use_raw=True).groupby("leiden").mean().mean(1)
        cols.append(s.rename(lin))
guide = pd.concat(cols, axis=1)
print("average broad-marker expression per cluster (higher = more like that lineage):")
print(guide.round(2).to_string())
best_guess = guide.idxmax(1).astype(str)
print("\nautomatic best guess per cluster:")
print(best_guess.to_string())

cluster_to_type = {          # EDIT: map every cluster number to a cell type, e.g. "0":"Fibroblast"
}
if not cluster_to_type:      # empty -> fall back to the best guess so the cell still runs
    cluster_to_type = best_guess.to_dict()

adata.obs["cell_type"] = adata.obs["leiden"].astype(str).map(cluster_to_type)
adata.obs["lineage"]   = adata.obs["cell_type"]        # a copy some later cells expect
assert adata.obs["cell_type"].notna().all(), "a cluster is unmapped: add it to cluster_to_type"
print("\ncell type composition:")
print(adata.obs["cell_type"].value_counts().to_string())
sc.pl.umap(adata, color="cell_type", frameon=False, title=f"{SAMPLE} cell types (generic annotation)")

**Cell types in space (a quick check).**

This plots every cell at its real position on the slide, coloured by the type you named in Step 9. The types
should form coherent tissue structures, not random confetti. Low quality cells are drawn faint so they do not
drown out the rest. The figure saves to your output folder. Nothing to edit here.

In [ ]:
# spatial map of the named cell types (the UMAP, but plotted on the tissue)
xy = adata.obsm["spatial"]
types = sorted(adata.obs["cell_type"].astype(str).unique())
cmap = plt.get_cmap("tab20").colors

fig, ax = plt.subplots(figsize=(12, 11))
for i, t in enumerate(types):
    m = (adata.obs["cell_type"].astype(str) == t).values
    color = "0.85" if t == "Low quality" else cmap[i % 20]   # low quality drawn faint grey
    ax.scatter(xy[m, 0], xy[m, 1], s=2, c=[color], label=f"{t} ({m.sum():,})", linewidths=0)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"{SAMPLE}: cell types in space")
ax.legend(markerscale=5, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(OUT / f"{SAMPLE}_celltypes_spatial.png", dpi=200, bbox_inches="tight")
plt.show()

# Part B. Optional worked example: keloid skin

**This whole part is an example, not a required step.** It shows a complete real analysis on keloid skin: naming
the clusters, splitting fibroblast states, a collagen control, cell type figures, plasma cell foci, and a
reference check. The **method** is reusable, but the **gene lists and the cluster to cell type maps are specific
to keloid skin**.

For your own tissue you have two clean options:
1. You already labelled your cells in Step 9. Skip to **Step 16 (BANKSY)** for tissue domains, then **Step 22** to
   save. You can still read the steps below as an example of what a deeper analysis looks like.
2. Or rewrite the pieces below for your biology: the markers, the `lineage_map` in Step 11, and the compartment
   you care about (here it is fibroblasts).

The cluster numbers from Leiden are not stable between runs, so any map keyed by a cluster number must be
rewritten every time you rerun the clustering.

## Step 10. Targeted marker check for specific clusters

A focused look at chosen markers across the clusters you are unsure about, to confirm what they are before
naming them. Change the gene list and the cluster numbers in `.loc[...]` to the clusters you want to inspect.

In [ ]:
check = ["COL1A1","DCN","PDGFRA",            # fibroblast
         "CD74","HLA-DRA","CIITA","CXCL12",  # MHC-II program
         "LYZ","CD68","C1QA","MRC1","AIF1","ITGAM","PTPRC",  # myeloid / leukocyte
         "MPZ","PMP22","SOX10",              # Schwann
         "PECAM1","KRT1","JCHAIN"]           # endothelial / keratinocyte / plasma
check = [g for g in check if g in adata.raw.var_names]
df = sc.get.obs_df(adata, keys=check + ["leiden"], use_raw=True).groupby("leiden").mean()
print(df.round(2).loc[["11","10","12","9","8","6"]].to_string())    # EDIT: the clusters you want to inspect

## Step 11. Name the clusters (rewrite this for every sample)

This is the manual annotation. `lineage_map` assigns each Leiden cluster number to a broad cell type based on
the markers from Steps 8 and 10. **You must edit this dictionary every run**, because cluster numbers change.
The assert at the end stops you if any cluster is left unmapped.

In [ ]:
lineage_map = {                        # EDIT: every cluster number must appear here
    # epidermis
    "0":"Keratinocyte", "1": "Keratinocyte", "2":"Keratinocyte","3":"Keratinocyte","4":"Keratinocyte",
    "5":"Keratinocyte","8":"Keratinocyte","20":"Melanocyte",
    # fibroblasts (11 = MHC-II fib with embedded macrophages, split later; 10/12 = low quality)
    "6":"Fibroblast","7":"Fibroblast","10":"Low quality","11":"Fibroblast",
    "12":"Low quality","13":"Fibroblast","14":"Fibroblast","15":"Fibroblast",
    "16":"Fibroblast","17":"Fibroblast","21":"Fibroblast",
    # vessels
    "18":"Lymphatic EC",
    # immune
    "19":"Plasma cell","9":"Plasma cell",
}
order = ["Keratinocyte","Melanocyte","Fibroblast","Lymphatic EC","Plasma cell","Low quality"]
cols = plt.get_cmap("tab20").colors
pal = {lin: cols[i % 20] for i, lin in enumerate(order)}
adata.obs["lineage"] = pd.Categorical(adata.obs["leiden"].astype(str).map(lineage_map), categories=order)
adata.uns["lineage_colors"] = [matplotlib.colors.to_hex(pal[c]) for c in order]
assert adata.obs["lineage"].isna().sum() == 0, "a cluster is unmapped, check keys"
print(adata.obs["lineage"].value_counts().to_string())
sc.pl.umap(adata, color="lineage", frameon=False, title=f"{SAMPLE} lineages")

Optional confirmation before splitting the fibroblasts: check endothelial and perivascular markers in the
fibroblast clusters, so you know which ones actually hide vessels. Read and continue.

In [ ]:
chk = ["PECAM1","VWF","CLDN5","KDR","EMCN",     # endothelial
       "ACTA2","TAGLN","MYH11","RGS5","NOTCH3",  # VSMC / pericyte
       "COL1A1","DCN","PDGFRA","CD74","CIITA"]    # fibroblast / MHC-II ref
chk = [g for g in chk if g in adata.raw.var_names]
df = sc.get.obs_df(adata, keys=chk + ["leiden"], use_raw=True).groupby("leiden").mean()
print(df.round(2).loc[["15","17","11","6"]].to_string())    # EDIT: fibroblast clusters to inspect

Tidy the lineage names for the figures. The cluster-to-lineage assignment does not change here, only the display
labels.

In [ ]:
rename = {"Keratinocyte":"Keratinocytes","Melanocyte":"Melanocytes","Fibroblast":"Fibroblasts",
          "Lymphatic EC":"Lymphatic endothelial cells","Plasma cell":"Plasma cells",
          "Low quality":"Low-quality (excluded)"}
lineage_map = {k: rename.get(v, v) for k, v in lineage_map.items()}
order = ["Keratinocytes","Melanocytes","Fibroblasts","Lymphatic endothelial cells",
         "Plasma cells","Low-quality (excluded)"]
cols = plt.get_cmap("tab20").colors
pal = {lin: cols[i % 20] for i, lin in enumerate(order)}
adata.obs["lineage"] = pd.Categorical(adata.obs["leiden"].astype(str).map(lineage_map), categories=order)
adata.uns["lineage_colors"] = [matplotlib.colors.to_hex(pal[c]) for c in order]
assert adata.obs["lineage"].isna().sum() == 0
print(adata.obs["lineage"].value_counts().to_string())

## Step 12. Fibroblast states (subclustering with collagen blocked)

Keloid is a fibrosis, so the fibroblasts are the interesting compartment. This takes only the fibroblasts,
recomputes variable genes on them, and this time **blocks collagen and other bulk matrix genes**. If collagen is
left in, the fibroblasts split by how much collagen they carry (a smooth intensity gradient) instead of by
function. Blocking it lets the functional states separate. The clean helper strips housekeeping genes from the
printed marker lists so the labels are readable.

In [ ]:
import re
def clean(genes, k=10):
    out = []
    for g in genes:
        gu = g.upper()
        if re.match(r'^(MT-|RP[SL]|LINC|MIR|AC\d|AL\d|AD\d|FP\d|AP\d{3})', gu): continue
        if gu in {"MALAT1","NEAT1","XIST","EEF1A1","TMSB4X","ACTB","B2M","TPT1","VIM"}: continue
        out.append(g)
        if len(out) == k: break
    return out

f = adata[adata.obs["lineage"] == "Fibroblasts"].copy()
f.X = f.layers["lognorm"].copy(); f.raw = f
sc.experimental.pp.highly_variable_genes(f, flavor="pearson_residuals", n_top_genes=2000, layer="counts")

vn = f.var_names.str.upper()
tech  = vn.str.startswith("MT-") | vn.str.match(r"^RP[SL]") | vn.str.match(r"^IG[HKL]") | vn.str.startswith("HB") | vn.isin(["MALAT1","NEAT1"])
ecm   = vn.str.startswith("COL") | vn.isin(["POSTN","SPARC","FN1","BGN","COMP","ASPN","AEBP1","TGFBI","CCDC80","MGP","FBN1","MMP2","IGF2"])   # matrix genes blocked here
epi   = vn.str.startswith("KRT") | vn.isin(["FLG","DSP","DSG1","DSG3","DSC2","DSC3","SBSN","DMKN","KRTDAP","PERP","SFN","CALML3"])            # epidermal ambient
noise = vn.str.match(r"^(AC|AL|AD|FP|AP)\d") | vn.str.startswith("LINC") | vn.str.startswith("HIST") | vn.isin(["KIRREL3","IL1RAPL2","KCNMA1","ROBO2","DLC1","NCKAP5"])   # clone loci + long-gene ambient
f.var["highly_variable"] &= ~(tech | ecm | epi | noise)
print("fibroblast variable genes after blocking:", int(f.var["highly_variable"].sum()))

adp = f[:, f.var["highly_variable"]].copy(); adp.X = adp.layers["counts"].copy()
sc.experimental.pp.normalize_pearson_residuals(adp)
sc.pp.pca(adp, n_comps=30, random_state=0)
f.obsm["X_pca"] = adp.obsm["X_pca"]
sc.pp.neighbors(f, n_neighbors=15, n_pcs=20)
sc.tl.leiden(f, resolution=0.3, key_added="sub", flavor="igraph", n_iterations=2, directed=False)
sc.tl.rank_genes_groups(f, "sub", method="wilcoxon")
fib = f

vc = fib.obs["sub"].value_counts()
names = pd.DataFrame(fib.uns["rank_genes_groups"]["names"])
print("fibroblast subclusters:", fib.obs["sub"].nunique())
for c in names.columns:
    print(f"fib_{c} (n={vc[c]:>5}): {', '.join(clean(names[c]))}")

The MHC-II-high fibroblast subcluster often has real macrophages embedded in it. This separates the two using a
macrophage score plus a hard count threshold, so the fibroblast state is not contaminated by myeloid cells.
Adjust the marker lists and the score/count thresholds if your tissue differs.

In [ ]:
mac = ["LYZ","CD68","C1QA","C1QB","AIF1","TYROBP","FCER1G","MRC1","ITGAM"]  # macrophage
fibm = ["COL1A1","DCN","PDGFRA","CIITA","CXCL12"]                            # fibroblast / HLA-DR program
g = [x for x in mac + fibm if x in fib.raw.var_names]

sub2 = fib[fib.obs["sub"] == "2"].copy()          # EDIT: the MHC-II-high subcluster number
sc.tl.score_genes(sub2, [x for x in mac if x in fib.raw.var_names], score_name="mac_score", use_raw=True)
print("macrophage-score distribution inside the MHC-II subcluster:")
print(sub2.obs["mac_score"].describe().round(3).to_string())

X = sub2[:, [x for x in mac if x in fib.var_names]].layers["counts"]
mac_counts = np.asarray(X.sum(1)).ravel() if sp.issparse(X) else np.asarray(X).sum(1)
is_mac = (sub2.obs["mac_score"] > 0.1) & (mac_counts >= 3)
print(f"\nlikely macrophages: {int(is_mac.sum()):,} of {sub2.n_obs:,}")
print(f"their median LYZ+CD68+C1QA counts: {np.median(mac_counts[is_mac.values]):.0f}  vs rest {np.median(mac_counts[~is_mac.values]):.0f}")
sub2.obs["call"] = np.where(is_mac.values, "macrophage", "HLA-DR+ fibroblast")
print("\n", sc.get.obs_df(sub2, keys=g + ["call"], use_raw=True).groupby("call").mean().round(2).T.to_string())

Name the fibroblast subclusters and push those labels back onto the whole tissue. Non-fibroblasts keep their
lineage; Schwann cells (which sit inside the fibroblast compartment by graph position) are corrected out. Edit
`fib_map` to match your subclusters. The spatial map is saved to the output folder.

In [ ]:
fib_map = {                            # EDIT: subcluster number ->fibroblast state
 "1":"Matrix/mechano",
 "0":"Perivascular/mural",
 "2":"HLA-DR+ fibroblast",
 "4":"Secretory-reticular",
 "8":"Schwann",                 # not a fibroblast, relabelled out
 "3":"unassigned","5":"unassigned","6":"unassigned","7":"unassigned","9":"unassigned",
}
fib_order = ["Matrix/mechano","Perivascular/mural","HLA-DR+ fibroblast",
             "Secretory-reticular","Schwann","unassigned"]
fib.obs["fib_type"] = pd.Categorical(fib.obs["sub"].map(fib_map), categories=fib_order)
print(fib.obs["fib_type"].value_counts().to_string())

adata.obs["cell_type"] = adata.obs["lineage"].astype(str)
adata.obs.loc[fib.obs_names, "cell_type"] = fib.obs["fib_type"].astype(str).values
adata.obs["cell_type"] = adata.obs["cell_type"].replace({"Fibroblasts":"Fibroblast (other)"})
print("\nfinal cell_type composition:")
print(adata.obs["cell_type"].value_counts().to_string())

xy = adata.obsm["spatial"]
show = ["Matrix/mechano","Perivascular/mural","HLA-DR+ fibroblast","Secretory-reticular","Schwann"]
cols = plt.get_cmap("tab10").colors
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(xy[:,0], xy[:,1], s=1, c="0.9", linewidths=0)
for i, t in enumerate(show):
    m = (adata.obs["cell_type"] == t).values
    if m.sum(): ax.scatter(xy[m,0], xy[m,1], s=3, c=[cols[i]], label=f"{t} ({m.sum():,})", linewidths=0)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(f"{SAMPLE} fibroblast subtypes")
ax.legend(markerscale=4, loc="center left", bbox_to_anchor=(1,0.5), frameon=False)
plt.tight_layout(); plt.savefig(OUT / f"{SAMPLE}_fib_subtypes.png", dpi=200, bbox_inches="tight"); plt.show()

## Step 13. Collagen kept versus blocked (the methods control)

This repeats the fibroblast subclustering but with collagen left in, to show what changes. With collagen kept,
the clusters line up with collagen intensity (a gradient). With collagen blocked (Step 12) the functional states
appear. The cross-tab prints how the two views correspond. This is the evidence that the states are real and not
a collagen artefact. It is a control, not a required output.

In [ ]:
fg = adata[adata.obs["lineage"] == "Fibroblasts"].copy()
fg.X = fg.layers["lognorm"].copy(); fg.raw = fg
sc.experimental.pp.highly_variable_genes(fg, flavor="pearson_residuals", n_top_genes=2000, layer="counts")
vn = fg.var_names.str.upper()
tech  = vn.str.startswith("MT-") | vn.str.match(r"^RP[SL]") | vn.str.match(r"^IG[HKL]") | vn.str.startswith("HB") | vn.isin(["MALAT1","NEAT1"])
epi   = vn.str.startswith("KRT") | vn.isin(["FLG","DSP","DSG1","DSG3","DSC2","DSC3","SBSN","DMKN","KRTDAP","PERP","SFN","CALML3"])
noise = vn.str.match(r"^(AC|AL|AD|FP|AP)\d") | vn.str.startswith("LINC") | vn.str.startswith("HIST") | vn.isin(["KIRREL3","IL1RAPL2","KCNMA1","ROBO2","DLC1","NCKAP5"])
fg.var["highly_variable"] &= ~(tech | epi | noise)      # note: NO ecm block, collagen stays in
print("gradient-pass variable genes (collagen KEPT):", int(fg.var["highly_variable"].sum()))

adp = fg[:, fg.var["highly_variable"]].copy(); adp.X = adp.layers["counts"].copy()
sc.experimental.pp.normalize_pearson_residuals(adp)
sc.pp.pca(adp, n_comps=30, random_state=0)
fg.obsm["X_pca"] = adp.obsm["X_pca"]
sc.pp.neighbors(fg, n_neighbors=15, n_pcs=20)
sc.tl.leiden(fg, resolution=0.3, key_added="sub", flavor="igraph", n_iterations=2, directed=False)
sc.tl.rank_genes_groups(fg, "sub", method="wilcoxon")

vc = fg.obs["sub"].value_counts()
names = pd.DataFrame(fg.uns["rank_genes_groups"]["names"])
print("\n--- collagen-kept subcluster markers ---")
for c in names.columns:
    print(f"grad_{c} (n={vc[c]:>5}): {', '.join(clean(names[c]))}")

colv = sc.get.obs_df(fg, keys=["COL1A1","COL3A1","POSTN","sub"], use_raw=True).groupby("sub").mean()
colv["n"] = fg.obs["sub"].value_counts()
print("\ngradient clusters ranked by collagen (proof it splits on matrix intensity):")
print(colv.sort_values("COL1A1", ascending=False).round(2).to_string())

fmap = {"1":"Matrix/mechano","0":"Perivascular/mural","2":"HLA-DR+ fib","4":"Secretory-reticular","8":"Schwann"}
common = fg.obs_names.intersection(fib.obs_names)
grad_lab = "grad_" + fg.obs.loc[common,"sub"].astype(str)
func_lab = fib.obs.loc[common,"sub"].astype(str).map(lambda s: fmap.get(s,"minor"))
print("\ncross-tab: gradient cluster (rows) vs functional label (cols)")
print(pd.crosstab(grad_lab, func_lab).to_string())

The side-by-side figure for the control: collagen intensity (before) next to functional states (after). Saved to
the output folder.

In [ ]:
fmap = {"1":"Matrix/mechano","0":"Perivascular/mural","2":"HLA-DR+ fibroblast","4":"Secretory-reticular","8":"Schwann"}
fib.obs["flabel"] = fib.obs["sub"].astype(str).map(lambda s: fmap.get(s, "minor"))

fig, ax = plt.subplots(1, 2, figsize=(20, 10))
xy = fg.obsm["spatial"]
col = sc.get.obs_df(fg, keys=["COL1A1"], use_raw=True)["COL1A1"].values
s = ax[0].scatter(xy[:,0], xy[:,1], s=3, c=col, cmap="viridis", linewidths=0)
ax[0].set_title("collagen kept: a smooth collagen-intensity gradient")
plt.colorbar(s, ax=ax[0], shrink=0.5, label="COL1A1")

order = ["Matrix/mechano","Perivascular/mural","HLA-DR+ fibroblast","Secretory-reticular","Schwann","minor"]
cols = plt.get_cmap("tab10").colors
xy2 = fib.obsm["spatial"]
ax[1].scatter(xy2[:,0], xy2[:,1], s=1, c="0.9", linewidths=0)
for i, t in enumerate(order):
    m = (fib.obs["flabel"] == t).values
    if m.sum() and t != "minor":
        ax[1].scatter(xy2[m,0], xy2[m,1], s=3, c=[cols[i]], label=f"{t} ({m.sum():,})", linewidths=0)
ax[1].set_title("collagen blocked: distinct functional states")
ax[1].legend(markerscale=4, loc="center left", bbox_to_anchor=(1,0.5), frameon=False)
for a in ax: a.invert_yaxis(); a.set_aspect("equal"); a.axis("off")
plt.tight_layout(); plt.savefig(OUT / f"{SAMPLE}_collagen_kept_vs_blocked.png", dpi=200, bbox_inches="tight"); plt.show()

## Step 14. Pull vessels and myeloid cells out of the fibroblast compartment

Segmentation puts some endothelial cells, mural cells, and myeloid cells inside fibroblast clusters. These two
cells reassign them using a score plus a hard raw-count requirement, so a cell is only moved if its defining
markers are genuinely present.

In [ ]:
def raw_counts(ad, genes):
    genes = [g for g in genes if g in ad.var_names]
    X = ad[:, genes].layers["counts"]
    return (np.asarray(X.sum(1)).ravel() if sp.issparse(X) else np.asarray(X).sum(1)), genes

fib_lab = {"1":"Fib: Matrix/mechano","0":"Fib: Perivascular","2":"Fib: MHC-II-high",
           "4":"Fib: Secretory-reticular","8":"Schwann"}
adata.obs["cell_type"] = adata.obs["lineage"].astype(str)
adata.obs.loc[fib.obs_names, "cell_type"] = fib.obs["sub"].astype(str).map(lambda s: fib_lab.get(s, "Fib: other")).values

for k, gs in {"fibro":["COL1A1","COL1A2","DCN","LUM","PDGFRA","FBLN1"],
              "bloodEC":["PECAM1","VWF","CLDN5","KDR","EMCN","CDH5","EGFL7"],
              "mural":["RGS5","NOTCH3","MYH11","PDGFRB","KCNJ8","ACTA2","TAGLN"]}.items():
    sc.tl.score_genes(adata, [g for g in gs if g in adata.raw.var_names], score_name=f"sc_{k}", use_raw=True)

ec_ct,  _ = raw_counts(adata, ["PECAM1","VWF","CLDN5","KDR","EMCN"])
mur_ct, _ = raw_counts(adata, ["RGS5","NOTCH3","MYH11","PDGFRB","KCNJ8"])
is_fibcomp = adata.obs["cell_type"].astype(str).str.startswith("Fib:").values
S = adata.obs[["sc_fibro","sc_bloodEC","sc_mural"]].values

new = adata.obs["cell_type"].astype(object).values.copy()
# require the defining markers to be genuinely present (>=3 raw counts) AND to win the score
ec_call  = is_fibcomp & (ec_ct  >= 3) & (S[:,1] > S[:,0]) & (S[:,1] >= S[:,2])
mur_call = is_fibcomp & (mur_ct >= 3) & (S[:,2] > S[:,0]) & (S[:,2] >  S[:,1]) & ~ec_call
new[ec_call]  = "Blood endothelial"
new[mur_call] = "Mural/pericyte"
adata.obs["cell_type"] = new
print(f"reclassified OUT of fibroblast: Blood endothelial {int(ec_call.sum()):,}, Mural/pericyte {int(mur_call.sum()):,}")
print("\nupdated composition:")
print(adata.obs["cell_type"].value_counts().to_string())

In [ ]:
mhc = (adata.obs["cell_type"] == "Fib: MHC-II-high").values
print(f"MHC-II-high compartment before separation: {int(mhc.sum()):,} cells\n")

sc.tl.score_genes(adata, [g for g in ["LYZ","CD68","C1QA","C1QB","AIF1","TYROBP","FCER1G","MRC1","ITGAM","LST1"]
                          if g in adata.raw.var_names], score_name="sc_myeloid", use_raw=True)
mye_ct, _ = raw_counts(adata, ["LYZ","CD68","C1QA","C1QB","AIF1","TYROBP"])
mye_call = mhc & (mye_ct >= 5) & (adata.obs["sc_myeloid"].values > adata.obs["sc_fibro"].values)
adata.obs.loc[mye_call, "cell_type"] = "Myeloid (uncertain)"
print(f"separated as Myeloid (uncertain): {int(mye_call.sum()):,} ({100*mye_call.sum()/mhc.sum():.1f}% of the compartment)")

rem = (adata.obs["cell_type"] == "Fib: MHC-II-high").values
sub = adata[rem]
print(f"\n--- remaining MHC-II-high fibroblasts: n = {int(rem.sum()):,} ---")
print("percent positive (raw counts > 0):")
for g in ["CIITA","CD74","HLA-DRA","HLA-DRB1","COL1A1","DCN"]:
    if g in adata.var_names:
        ct,_ = raw_counts(sub,[g]); print(f"  {g:9s}: {100*(ct>0).mean():5.1f}%  (identity, want high)")
for g in ["PTPRC","LST1","TYROBP","FCER1G","AIF1","C1QA","C1QB","LYZ","CD68"]:
    if g in adata.var_names:
        ct,_ = raw_counts(sub,[g]); print(f"  {g:9s}: {100*(ct>0).mean():5.1f}%  (myeloid, want low)")
print(f"  median volume {adata.obs.loc[rem,'volume'].median():.0f} (all {adata.obs['volume'].median():.0f}) | "
      f"median counts {adata.obs.loc[rem,'total_counts'].median():.0f} (all {adata.obs['total_counts'].median():.0f})")

In [ ]:
cf = adata[adata.obs["cell_type"].astype(str).str.startswith("Fib:")].copy()
cf.X = cf.layers["lognorm"].copy(); cf.raw = cf
sc.experimental.pp.highly_variable_genes(cf, flavor="pearson_residuals", n_top_genes=2000, layer="counts")
vn = cf.var_names.str.upper()
block = (vn.str.startswith("MT-")|vn.str.match(r"^RP[SL]")|vn.str.match(r"^IG[HKL]")|vn.str.startswith("HB")|vn.isin(["MALAT1","NEAT1"])
         |vn.str.startswith("COL")|vn.isin(["POSTN","SPARC","FN1","BGN","COMP","ASPN","AEBP1","TGFBI","CCDC80","MGP","FBN1","MMP2","IGF2"])
         |vn.str.startswith("KRT")|vn.str.match(r"^(AC|AL|AD|FP|AP)\d")|vn.str.startswith("LINC")|vn.str.startswith("HIST")
         |vn.str.startswith("HLA-")|vn.isin(["CD74","CIITA","B2M","TMSB4X"]))     # MHC-II EXCLUDED from features
cf.var["highly_variable"] &= ~block
adp = cf[:, cf.var["highly_variable"]].copy(); adp.X = adp.layers["counts"].copy()
sc.experimental.pp.normalize_pearson_residuals(adp); sc.pp.pca(adp, n_comps=30, random_state=0)
cf.obsm["X_pca"] = adp.obsm["X_pca"]; sc.pp.neighbors(cf, n_neighbors=15, n_pcs=20)
sc.tl.leiden(cf, resolution=0.3, key_added="sub2", flavor="igraph", n_iterations=2, directed=False)
sc.tl.score_genes(cf, [g for g in ["HLA-DRA","HLA-DRB1","HLA-DPB1","HLA-DQB1","CD74","CIITA"] if g in cf.raw.var_names],
                  score_name="mhc2_score", use_raw=True)
tab = cf.obs.groupby("sub2")["mhc2_score"].agg(["mean","count"]).round(3).sort_values("mean", ascending=False)
print("MHC-II score per subcluster, where clustering used NO MHC-II genes:")
print(tab.to_string())

## Step 15. Cell type figures

A marker dotplot per cell type, and the spatial map of all cell types.
`markers` lists the canonical genes per type; `order` fixes the row order so the dotplot reads logically. Both
figures are saved to the output folder. For a new tissue, edit `markers` and `order`.

In [ ]:
markers = {
 "Fibroblast (matrix)":   ["COL1A1","COL3A1","POSTN","ADAM12","ASPN"],
 "Perivascular fib":      ["COL4A1","AQP1","A2M","IGFBP7"],
 "MHC-II-high fib":       ["CD74","HLA-DRA","HLA-DRB1","CIITA","CXCL12"],
 "Secretory-reticular":   ["PI16","CXCL14","CFD","TNXB"],
 "Mural/pericyte":        ["RGS5","NOTCH3","MYH11","PDGFRB"],
 "Blood endothelial":     ["PECAM1","VWF","CLDN5","KDR"],
 "Lymphatic EC":          ["CCL21","MMRN1","PROX1","LYVE1"],
 "Keratinocyte":          ["KRT1","KRT10","KRT14","FLG"],
 "Melanocyte":            ["MLANA","PMEL","DCT","TYR"],
 "Plasma cell":           ["JCHAIN","IGKC","MZB1"],
 "Schwann":               ["MPZ","PMP22","GPM6B"],
 "Myeloid":               ["LYZ","CD68","C1QA","PTPRC"],
}
markers = {k:[g for g in v if g in adata.raw.var_names] for k,v in markers.items()}
order = ["Keratinocytes","Melanocytes","Fib: Matrix/mechano","Fib: Perivascular","Fib: MHC-II-high",
         "Fib: Secretory-reticular","Fib: other","Mural/pericyte","Blood endothelial",
         "Lymphatic endothelial cells","Plasma cells","Schwann","Myeloid (uncertain)"]
order = [o for o in order if o in adata.obs["cell_type"].unique()]
adata.obs["cell_type"] = pd.Categorical(adata.obs["cell_type"], categories=order)
sc.pl.dotplot(adata, markers, groupby="cell_type", standard_scale="var", dendrogram=False, figsize=(20,7))

In [ ]:
xy = adata.obsm["spatial"]
show = ["Keratinocytes","Melanocytes","Fib: Matrix/mechano","Fib: Perivascular","Fib: MHC-II-high",
        "Fib: Secretory-reticular","Mural/pericyte","Blood endothelial","Lymphatic endothelial cells",
        "Plasma cells","Schwann","Myeloid (uncertain)"]
cmap = plt.get_cmap("tab20").colors
fig, ax = plt.subplots(figsize=(12, 11))
ax.scatter(xy[:,0], xy[:,1], s=1, c="0.92", linewidths=0)
for i, t in enumerate(show):
    m = (adata.obs["cell_type"] == t).values
    if m.sum(): ax.scatter(xy[m,0], xy[m,1], s=3, c=[cmap[i % 20]], label=f"{t} ({m.sum():,})", linewidths=0)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(f"{SAMPLE}: all cell types")
ax.legend(markerscale=4, loc="center left", bbox_to_anchor=(1,0.5), frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(OUT / f"{SAMPLE}_all_celltypes.png", dpi=200, bbox_inches="tight"); plt.show()

## Step 16. BANKSY tissue domains

BANKSY groups cells into spatial domains by mixing each cell's own expression with its neighbourhood. The
`lambda` value sets the balance: low lambda (0.2) gives smoothed cell states, high lambda (0.8) gives tissue
domains. This runs a primary pass with collagen kept and a sensitivity pass with collagen removed, so you can
check the domains are not just a collagen map. This step is heavier and takes a few minutes. It needs the
`banksy` environment. Change `res` for more or fewer domains.

In [ ]:
# BANKSY for keloid (For other non-keloid samples, see below)

from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition
from scipy.spatial import cKDTree

tree = cKDTree(adata.obsm["spatial"]); d,_ = tree.query(adata.obsm["spatial"], k=19)
print(f"k_geom=18 radius: median {np.median(d[:,18]):.1f} um, 5-95th {np.percentile(d[:,18],5):.1f}-{np.percentile(d[:,18],95):.1f} um")

adata.obs["x"] = adata.obsm["spatial"][:,0]; adata.obs["y"] = adata.obsm["spatial"][:,1]
coord_keys = ("x","y","spatial")

def run_banksy(mask, tag, lambdas, k_geom=18, pca_dims=20, res=1.0, seed=0):
    bank = adata[:, mask].copy()
    bank.X = bank.layers["lognorm"].copy(); sc.pp.scale(bank, max_value=10)
    for lam in lambdas:
        bd = initialize_banksy(bank, coord_keys, k_geom, nbr_weight_decay="scaled_gaussian", max_m=1,
                               plt_edge_hist=False, plt_nbr_weights=False, plt_agf_angles=False, plt_theta=False)
        bd,_ = generate_banksy_matrix(bank, bd, [lam], 1)
        pca_umap(bd, pca_dims=[pca_dims], add_umap=False, plt_remaining_var=False)
        rdf,_ = run_Leiden_partition(bd, resolutions=[res], num_nn=50, match_labels=False, partition_seed=seed)
        key = f"banksy_{tag}_l{int(lam*10):02d}"
        adata.obs[key] = pd.Categorical(np.asarray(rdf.loc[rdf.index[0],"labels"].dense).astype(str))
        print(f"{key}: lambda={lam} -> {adata.obs[key].nunique()} domains")

# PRIMARY: collagen kept in
run_banksy(adata.var["highly_variable"].values, "primary", lambdas=(0.2, 0.8))

# SENSITIVITY: collagen/ECM removed, lambda 0.8 only
vn = adata.var_names.str.upper()
ecm = vn.str.startswith("COL") | vn.isin(["POSTN","SPARC","FN1","BGN","COMP","ASPN","AEBP1","TGFBI","CCDC80","MGP","FBN1","MMP2","IGF2"])
run_banksy(adata.var["highly_variable"].values & ~np.asarray(ecm), "ecmreduced", lambdas=(0.8,))

Draw the BANKSY domains: the two lambdas of the primary run, and primary against sensitivity at lambda 0.8. If
the primary and sensitivity domains look the same, the domains are collagen-independent. Figures saved.

In [ ]:
def plot_domains(key, ax, title):
    xy = adata.obsm["spatial"]
    cats = adata.obs[key].cat.categories
    cmap = plt.get_cmap("tab20").colors
    for i, dd in enumerate(cats):
        m = (adata.obs[key] == dd).values
        ax.scatter(xy[m,0], xy[m,1], s=2, c=[cmap[i % 20]], label=dd, linewidths=0)
    ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)
    ax.legend(markerscale=5, loc="center left", bbox_to_anchor=(1,0.5), fontsize=6, ncol=2, frameon=False)

fig, ax = plt.subplots(1, 2, figsize=(22, 10))
plot_domains("banksy_primary_l02", ax[0], f"{SAMPLE}: lambda=0.2 (smoothed cell states)")
plot_domains("banksy_primary_l08", ax[1], f"{SAMPLE}: lambda=0.8 (tissue domains)")
plt.tight_layout(); plt.savefig(OUT / f"{SAMPLE}_banksy_primary.png", dpi=200, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(22, 10))
plot_domains("banksy_primary_l08",    ax[0], f"{SAMPLE}: lambda=0.8 PRIMARY (collagen in)")
plot_domains("banksy_ecmreduced_l08", ax[1], f"{SAMPLE}: lambda=0.8 SENSITIVITY (collagen out)")
plt.tight_layout(); plt.savefig(OUT / f"{SAMPLE}_banksy_primary_vs_sensitivity.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
#Generic BANKSY (for any other samples)

# Groups cells into spatial domains by mixing each cell's expression with its neighbourhood.
# lambda 0.2 = smoothed cell states, lambda 0.8 = broad tissue domains.
# Needs Steps 5 and 6 done: adata.var["highly_variable"], adata.layers["lognorm"], adata.obsm["spatial"].

from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition

LAMBDAS  = (0.2, 0.8)   # EDIT: which lambdas to run
K_GEOM   = 18           # neighbours used to define each cell's neighbourhood
PCA_DIMS = 20
RES      = 1.0          # EDIT: higher = more domains

adata.obs["x"] = adata.obsm["spatial"][:, 0]
adata.obs["y"] = adata.obsm["spatial"][:, 1]
coord_keys = ("x", "y", "spatial")

# use the highly variable genes on the log-normalised values, scaled
bank = adata[:, adata.var["highly_variable"].values].copy()
bank.X = bank.layers["lognorm"].copy()
sc.pp.scale(bank, max_value=10)

for lam in LAMBDAS:
    bd = initialize_banksy(bank, coord_keys, K_GEOM, nbr_weight_decay="scaled_gaussian", max_m=1,
                           plt_edge_hist=False, plt_nbr_weights=False, plt_agf_angles=False, plt_theta=False)
    bd, _ = generate_banksy_matrix(bank, bd, [lam], 1)
    pca_umap(bd, pca_dims=[PCA_DIMS], add_umap=False, plt_remaining_var=False)
    rdf, _ = run_Leiden_partition(bd, resolutions=[RES], num_nn=50, match_labels=False, partition_seed=0)
    key = f"banksy_primary_l{int(lam*10):02d}"       # names: banksy_primary_l02, banksy_primary_l08
    adata.obs[key] = pd.Categorical(np.asarray(rdf.loc[rdf.index[0], "labels"].dense).astype(str))
    print(f"{key}: lambda={lam} -> {adata.obs[key].nunique()} domains")

In [ ]:
# Spatial plot the BANKSY domains 
def plot_domains(key, ax, title):
    xy = adata.obsm["spatial"]
    cmap = plt.get_cmap("tab20").colors
    for i, dd in enumerate(adata.obs[key].cat.categories):
        m = (adata.obs[key] == dd).values
        ax.scatter(xy[m, 0], xy[m, 1], s=2, c=[cmap[i % 20]], label=dd, linewidths=0)
    ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)
    ax.legend(markerscale=5, loc="center left", bbox_to_anchor=(1, 0.5), fontsize=7, ncol=2, frameon=False)

fig, ax = plt.subplots(1, 2, figsize=(22, 10))
plot_domains("banksy_primary_l02", ax[0], f"{SAMPLE}: lambda=0.2 (smoothed cell states)")
plot_domains("banksy_primary_l08", ax[1], f"{SAMPLE}: lambda=0.8 (tissue domains)")
plt.tight_layout()
plt.savefig(OUT / f"{SAMPLE}_banksy_domains.png", dpi=200, bbox_inches="tight")
plt.show()

## Step 17. What each domain is made of, and name the domains

This tabulates the cell type composition of each BANKSY domain and the enrichment of the key populations, then
maps the numbered domains to named tissue regions. **Edit `domain_map`** to match your domains and your biology
(as with the cluster map, the numbers are not stable between runs). The region map figure is saved.

In [ ]:
key = "banksy_primary_l08"
comp = pd.crosstab(adata.obs[key], adata.obs["cell_type"], normalize="index")
n_dom = adata.obs[key].value_counts()
from scipy.stats import entropy
div = comp.apply(lambda r: entropy(r + 1e-9), axis=1)

print("domain composition (top 3 cell types each):")
for dd in comp.index:
    top3 = (comp.loc[dd].sort_values(ascending=False).head(3) * 100).round(0)
    print(f"  domain {dd:>2} (n={n_dom[dd]:>6,}, mixing {div[dd]:.2f}): " +
          ", ".join(f"{k} {v:.0f}%" for k, v in top3.items()))

print("\n--- enrichment (fold over tissue average) for the key populations ---")
for ct in ["Fib: MHC-II-high","Plasma cells","Blood endothelial","Lymphatic endothelial cells",
           "Mural/pericyte","Fib: Matrix/mechano"]:
    if ct in comp.columns:
        base = (adata.obs["cell_type"] == ct).mean()
        enr = (comp[ct] / base).sort_values(ascending=False).head(3)
        print(f"\n{ct} (tissue avg {base*100:.1f}%):")
        for dd, v in enr.items():
            print(f"    domain {dd:>2}: {v:4.1f}x  ({comp.loc[dd, ct]*100:.1f}% of domain)")

In [ ]:
domain_map = {                         # EDIT: domain number -> named region
    "0":"Immune-vascular dermis", "2":"Vascular/perivascular", "14":"Lymphatic",
    "5":"Matrix core","6":"Matrix core","8":"Matrix core",
    "1":"Matrix-perivascular","15":"Immune-stromal interface",
    "3":"Epidermis","7":"Epidermis","10":"Epidermis","11":"Epidermis",
    "12":"Epidermis","13":"Epidermis","16":"Epidermis",
    "4":"Dermal-epidermal junction","9":"Low-content dermis",
}
adata.obs["region"] = adata.obs["banksy_primary_l08"].astype(str).map(domain_map)
print(adata.obs["region"].value_counts().to_string())

xy = adata.obsm["spatial"]
regions = adata.obs["region"].value_counts().index.tolist()
cmap = plt.get_cmap("tab20").colors
fig, ax = plt.subplots(figsize=(12, 11))
for i, r in enumerate(regions):
    m = (adata.obs["region"] == r).values
    ax.scatter(xy[m,0], xy[m,1], s=2, c=[cmap[i % 20]], label=f"{r} ({m.sum():,})", linewidths=0)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(f"{SAMPLE}: BANKSY tissue domains (lambda=0.8)")
ax.legend(markerscale=5, loc="center left", bbox_to_anchor=(1,0.5), frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(OUT / f"{SAMPLE}_regions.png", dpi=200, bbox_inches="tight"); plt.show()

## Step 18. Which cell types touch each other more than chance (niche test)

This builds a contact graph from the cell positions (Delaunay triangulation, pruned to real contacts within 30
microns), then asks whether chosen cell type pairs touch more than expected. The permutation test shuffles the
labels many times to build the null. It reports the global z-score and, more conservatively, the z-score when
shuffling only within each region (so a pair is not called just because both types share a region). Edit `pairs`
to the interactions you care about. This is a slow cell.

In [ ]:
from scipy.spatial import Delaunay

xy = adata.obsm["spatial"]
tri = Delaunay(xy)
e = np.vstack([tri.simplices[:, [0,1]], tri.simplices[:, [1,2]], tri.simplices[:, [0,2]]])
e = np.unique(np.sort(e, axis=1), axis=0)
d = np.sqrt(((xy[e[:,0]] - xy[e[:,1]])**2).sum(1))
edges = e[d <= 30]        # EDIT: contact distance in microns
print(f"{len(edges):,} contact edges (median length {np.median(d[d<=30]):.1f} um)")

lab = adata.obs["cell_type"].astype(str).values
reg = adata.obs["region"].astype(str).values

def contacts(labels, a, b):
    la, lb = labels[edges[:,0]], labels[edges[:,1]]
    return int(((la==a)&(lb==b)).sum() + ((la==b)&(lb==a)).sum())

def perm_test(a, b, within_region, n=1000, seed=0):
    obs = contacts(lab, a, b)
    rng = np.random.default_rng(seed)
    null = np.empty(n)
    if within_region:
        idx_by_reg = {r: np.where(reg==r)[0] for r in np.unique(reg)}
    for i in range(n):
        perm = lab.copy()
        if within_region:
            for r, idx in idx_by_reg.items():
                perm[idx] = lab[rng.permutation(idx)]
        else:
            perm = lab[rng.permutation(len(lab))]
        null[i] = contacts(perm, a, b)
    z = (obs - null.mean()) / (null.std() + 1e-9)
    p = (np.sum(null >= obs) + 1) / (n + 1)
    return obs, null.mean(), z, p

pairs = [("Fib: MHC-II-high","Plasma cells"),          # EDIT: the interactions you care about
         ("Fib: MHC-II-high","Blood endothelial"),
         ("Fib: MHC-II-high","Fib: Matrix/mechano"),
         ("Plasma cells","Plasma cells"),
         ("Fib: Matrix/mechano","Blood endothelial")]

print(f"\n{'pair':<48}{'obs':>7}{'null':>8}{'z(global)':>11}{'z(within-region)':>18}")
for a, b in pairs:
    _,_, zg, _ = perm_test(a, b, within_region=False)
    obs, exp, zw, pw = perm_test(a, b, within_region=True)
    print(f"{a+' <-> '+b:<48}{obs:>7}{exp:>8.0f}{zg:>11.1f}{zw:>18.1f}")

## Step 19. Publication figure panels

The clean annotation figures: a cell-type UMAP and a top-marker matrixplot, both computed on a filtered copy
that drops the low-quality and uncertain buckets so the figure shows only confident cell types. Then marker
UMAPs and a fibroblast-only UMAP. All save into the output folder. Run the ones you want.

In [ ]:
# build a clean, ordered subset for the figures (drop low-quality / unknown / uncertain)
adata.obs["cell_type"] = adata.obs["cell_type"].astype(object).fillna("Unknown/low-confidence")
drop = adata.obs["cell_type"].str.contains("Low-|Unknown|uncertain|unresolved", case=False)
plot = adata[~drop.values].copy()
order = ["Keratinocytes","Melanocytes","Fib: Matrix/mechano","Fib: Perivascular","Fib: MHC-II-high",
         "Fib: Secretory-reticular","Mural/pericyte","Blood endothelial",
         "Lymphatic endothelial cells","Plasma cells","Schwann"]
order = [c for c in order if c in plot.obs["cell_type"].unique()]
plot.obs["cell_type"] = pd.Categorical(plot.obs["cell_type"], categories=order)
plot = plot[plot.obs["cell_type"].notna().values].copy()
plot.obs["cell_type"] = plot.obs["cell_type"].cat.remove_unused_categories()

# panel: annotated cell-type UMAP
sc.pl.umap(plot, color="cell_type", frameon=True, legend_loc="right margin",
           title=f"{SAMPLE}", size=8, palette="tab20", save=f"_{SAMPLE}_celltypes.png")

# panel: top markers per cell type as an evenly spaced matrixplot (reads cleanly, no label overlap)
sc.tl.rank_genes_groups(plot, "cell_type", method="wilcoxon", use_raw=True)
sc.pl.rank_genes_groups_matrixplot(
    plot, n_genes=4, use_raw=True, cmap="viridis", standard_scale="var",
    dendrogram=False, figsize=(16, 7), save=f"_{SAMPLE}_markers.png")

In [ ]:
# marker feature UMAPs on the full object
feats = ["KRT1","MLANA","DCN","CD74","CIITA","PI16","RGS5","PECAM1","CCL21","JCHAIN","MPZ","LYZ"]
feats = [g for g in feats if g in adata.raw.var_names]
sc.pl.umap(adata, color=feats, use_raw=True, cmap="Blues", frameon=False, ncols=4, size=5,
           save=f"_{SAMPLE}_markers_umap.png")

In [ ]:
# fibroblast-only UMAP, coloured by final state
sc.tl.umap(fib)
fib.obs["cell_type_final"] = adata.obs.loc[fib.obs_names, "cell_type"].astype(str).values
keep = ~fib.obs["cell_type_final"].str.contains("Low-|Unknown|uncertain")
sc.pl.umap(fib[keep], color="cell_type_final", frameon=True, legend_loc="right margin",
           size=6, palette="tab10", title=f"{SAMPLE}: fibroblast states",
           save=f"_{SAMPLE}_fib_umap.png")

## Step 20. Plasma-cell foci and isotype (keloid-specific example)

An example of a focused spatial question: are plasma cells scattered or clustered into foci, and which antibody
class dominates. It uses DBSCAN to find groups of plasma cells within 30 microns (at least 5 per focus) and sums
the immunoglobulin heavy-chain counts by isotype. Swap the cell type and gene sets for a different question.

In [ ]:
from sklearn.cluster import DBSCAN

pc = adata[adata.obs["cell_type"] == "Plasma cells"].copy()
xy = pc.obsm["spatial"]

pc.obs["focus"] = DBSCAN(eps=30, min_samples=5).fit(xy).labels_    # EDIT: eps (um) and min_samples per focus
sizes = pd.Series(pc.obs["focus"]).value_counts().drop(-1, errors="ignore")
in_foci = (pc.obs["focus"] >= 0).values
print(f"{pc.n_obs} plasma cells: {len(sizes)} foci, {int(in_foci.sum())} in foci ({100*in_foci.mean():.0f}%), {int((~in_foci).sum())} scattered")
print("largest foci (cells):", sorted(sizes.values, reverse=True)[:8])

def gsum(ad, genes):
    genes = [g for g in genes if g in ad.var_names]
    if not genes: return np.zeros(ad.n_obs)
    X = ad[:, genes].layers["counts"]
    return np.asarray(X.sum(1)).ravel() if sp.issparse(X) else np.asarray(X).sum(1)
for k, gs in {"IgG":["IGHG1","IGHG2","IGHG3","IGHG4"],"IgA":["IGHA1","IGHA2"],"IgM":["IGHM"]}.items():
    pc.obs[k] = gsum(pc, gs)
tot = pc.obs[["IgG","IgA","IgM"]].sum()
print("\nisotype (% of Ig heavy-chain counts):")
print((tot / tot.sum() * 100).round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 9))
allxy = adata.obsm["spatial"]
ax.scatter(allxy[:,0], allxy[:,1], s=1, c="0.9", linewidths=0)
ax.scatter(xy[~in_foci,0], xy[~in_foci,1], s=8, c="0.6", linewidths=0, label="scattered")
ax.scatter(xy[in_foci,0],  xy[in_foci,1],  s=14, c="crimson", linewidths=0, label=f"in foci ({int(in_foci.sum())})")
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(f"{SAMPLE}: plasma-cell foci")
ax.legend(markerscale=2, loc="center left", bbox_to_anchor=(1,0.5), frameon=False)
plt.tight_layout(); plt.savefig(OUT / f"{SAMPLE}_plasma_foci.png", dpi=200, bbox_inches="tight"); plt.show()

## Step 21. Optional: check labels against a reference atlas

If you have an annotated reference for this tissue, this trains a simple classifier on it and transfers labels to
your data as an independent check on the manual annotation. It first validates the classifier on the reference by
leaving one donor out, then transfers with a confidence threshold, then cross-tabulates your labels against the
transferred ones. **This needs a matching reference `.h5ad` and is keloid-specific here.** Skip it if you have no
reference. Edit the reference path, the `broad` label map, and `q_broad` for your data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

ref = sc.read_h5ad("/Users/chobani/Downloads/keloid_ref_mast 1.h5ad")     # EDIT: your reference atlas
ref.raw = None; ref.var_names_make_unique(); ref.layers["counts"] = ref.X.copy()
broad = {"Myofibro_Keloid":"Fibroblast","Myofibro_Normal":"Fibroblast","Endothelial":"Endothelial",
 "SMC":"SMC/perivascular","Lymphatics":"Lymphatic EC","KRT1_KC":"Keratinocyte","KRT14_KC":"Keratinocyte",
 "Lyve1pos_Macs":"Macrophage","Lyve1neg_Macs":"Macrophage","T_cell_Keloid":"T cell","T_cell_Scar":"T cell",
 "CD8_EM_TRMs":"T cell","CD8_EM_TRMs_act":"T cell","Mast Cells":"Mast","Schwann Cells":"Schwann",
 "Melanocytes":"Melanocyte","RBCs":"RBC","Sweat_Ducts":"Sweat duct"}
ref.obs["broad"] = ref.obs["manual_annot"].map(broad)

feat = [g for g in adata.var_names[adata.var["highly_variable"]] if g in set(ref.var_names)]
print("shared features:", len(feat))

def mat(ad):
    m = ad[:, feat].copy(); m.X = m.layers["counts"].copy()
    sc.pp.normalize_total(m, target_sum=1e4); sc.pp.log1p(m); sc.pp.scale(m, max_value=10)
    return np.asarray(m.X)

drop = ["RBC","Sweat duct"]
keep = (~ref.obs["broad"].isin(drop) & ref.obs["broad"].notna()).values
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(keep & (ref.obs["broad"]==c).values)[0],
        size=min(2000, int((keep & (ref.obs["broad"]==c).values).sum())), replace=False)
        for c in ref.obs.loc[keep,"broad"].unique()])
Xr = mat(ref[idx]); yr = ref.obs["broad"].values[idx].astype(str); pat = ref.obs["orig.ident"].values[idx].astype(str)

f1s = []
for p in np.unique(pat):
    tr, te = pat!=p, pat==p
    c = LogisticRegression(max_iter=1500, C=0.1, class_weight="balanced", n_jobs=-1).fit(Xr[tr], yr[tr])
    f1s.append(pd.Series(dict(zip(np.unique(yr), f1_score(yr[te], c.predict(Xr[te]), average=None, labels=np.unique(yr))))))
print("\nleave-one-donor-out mean F1 per class:")
print(pd.concat(f1s,axis=1).mean(1).sort_values(ascending=False).round(2).to_string())

clf = LogisticRegression(max_iter=2000, C=0.1, class_weight="balanced", n_jobs=-1).fit(Xr, yr)
proba = clf.predict_proba(mat(adata)); classes = clf.classes_
adata.obs["ref_pred"] = classes[proba.argmax(1)]
adata.obs["ref_conf"] = proba.max(1)
adata.obs.loc[adata.obs["ref_conf"] < 0.5, "ref_pred"] = "Unknown"

q_broad = {"Keratinocytes":"Keratinocyte","Melanocytes":"Melanocyte","Fib: Matrix/mechano":"Fibroblast",
 "Fib: Perivascular":"Fibroblast","Fib: MHC-II-high":"Fibroblast","Fib: Secretory-reticular":"Fibroblast",
 "Mural/pericyte":"SMC/perivascular","Blood endothelial":"Endothelial",
 "Lymphatic endothelial cells":"Lymphatic EC","Plasma cells":"Plasma","Schwann":"Schwann","Myeloid (uncertain)":"Macrophage"}
adata.obs["our_broad"] = adata.obs["cell_type"].map(q_broad)
m = adata.obs["our_broad"].notna()
ct = pd.crosstab(adata.obs.loc[m,"our_broad"], adata.obs.loc[m,"ref_pred"], normalize="index")*100
print("\nour annotation (rows) vs reference transfer (cols), % of row:")
print(ct.round(0).to_string())

## Step 22. Save the annotated object

Writes the fully annotated `.h5ad` (cell types, regions, all the QC and score columns) so the analysis does not
have to be rerun and can be shared. This is the end of the pipeline.

In [ ]:
# quick check: counts are still integers, X is log-normalised
c = adata.layers["counts"]
chk = c[:100].toarray() if sp.issparse(c) else np.asarray(c[:100])
print("counts.max():", c.max(), "| integer-valued?", np.allclose(chk, np.round(chk)))
print("X.max()     :", adata.X.max(), "(this is lognorm, expect ~6-8)")

FINAL = OUT / f"{SAMPLE}_annotated.h5ad"
adata.write_h5ad(FINAL, compression="gzip")
print("\nsaved:", FINAL, "| size MB:", round(FINAL.stat().st_size / 1e6, 1))
print("figures are in:", OUT)